In [ ]:
# Uncomment in a fresh environment:
# %pip install -q -r requirements.txt

import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from tqdm.auto import tqdm

print("torch:", torch.__version__)
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = "./data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

# Saved during training when validation accuracy improves
CKPT_PATH = "./checkpoints/best_model_valacc.pt"
# Our submitted model (phase 2, epoch 21, 71% val / 76% Kaggle)
SUBMIT_CKPT_PATH = "./checkpoints/best_model_On_Kaggle.pt"
SUBMISSION_OUT = "./submission.csv"

IMG_SIZE = 224
NUM_CLASSES = 100
BATCH_SIZE = 16
NUM_WORKERS = 0

PHASE1_EPOCHS = 6
PHASE2_EPOCHS = 50
PHASE1_LR = 8e-4
PHASE2_LR = 6e-5
PHASE2_HEAD_LR_MULT = 5

WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA = 0.15
CUTMIX_ALPHA = 0.3
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 12

CROP_PADDING = 48
RANDOM_ERASING_P = 0.2
HEAD_HIDDEN_DIM = 512

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def set_seed(seed: int = 82):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(82)
os.makedirs("./checkpoints", exist_ok=True)


## Data

Train images live in class folders `0`–`99`. We hold out one image per class for validation (100 images total) and augment only the training split.


In [ ]:
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + CROP_PADDING, IMG_SIZE + CROP_PADDING)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.12)),
])

train_tf = aug_tf
val_tf = eval_tf


class TrainImageDataset(Dataset):
    """Load images from folders named 0..99; label = folder number."""

    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []

        class_dirs = sorted(
            (d for d in os.scandir(root) if d.is_dir() and d.name.isdigit()),
            key=lambda d: int(d.name),
        )
        for cls_dir in class_dirs:
            label = int(cls_dir.name)
            for entry in sorted(os.scandir(cls_dir.path), key=lambda e: e.name):
                if entry.name.lower().endswith(".jpg"):
                    self.samples.append((entry.path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


class ApplyTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def stratified_split(samples, val_per_class=1, seed=82):
    rng = random.Random(seed)
    by_label = {}
    for idx, (_, label) in enumerate(samples):
        by_label.setdefault(label, []).append(idx)

    train_idx, val_idx = [], []
    for label in sorted(by_label):
        indices = by_label[label][:]
        rng.shuffle(indices)
        n_val = min(val_per_class, max(1, len(indices) // 5))
        val_idx.extend(indices[:n_val])
        train_idx.extend(indices[n_val:])
    return train_idx, val_idx


full_train = TrainImageDataset(TRAIN_DIR)
assert len({lbl for _, lbl in full_train.samples}) == NUM_CLASSES

train_idx, val_idx = stratified_split(full_train.samples, val_per_class=1, seed=82)

train_set = ApplyTransform(Subset(full_train, train_idx), train_tf)
val_set = ApplyTransform(Subset(full_train, val_idx), val_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"train: {len(train_set)} | val: {len(val_set)}")
print("train dir exists:", os.path.isdir(TRAIN_DIR))


## Model

In [ ]:
from torchvision.models import EfficientNet_V2_S_Weights

weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = models.efficientnet_v2_s(weights=weights)


def set_requires_grad(module, requires_grad: bool):
    for p in module.parameters():
        p.requires_grad = requires_grad


def freeze_efficientnet_features(model):
    """Phase 1: ImageNet features frozen; only classifier trains."""
    set_requires_grad(model.features, False)
    set_requires_grad(model.classifier, True)


def unfreeze_efficientnet_all(model):
    """Phase 2: full fine-tune — all features + head."""
    set_requires_grad(model.features, True)
    set_requires_grad(model.classifier, True)


def build_efficientnet_head(in_features, num_classes, hidden_dim=HEAD_HIDDEN_DIM):
    return nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, hidden_dim),
        nn.SiLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(hidden_dim, num_classes),
    )


in_features = model.classifier[1].in_features
model.classifier = build_efficientnet_head(in_features, NUM_CLASSES)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
freeze_efficientnet_features(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EfficientNet-V2-S | params: {total:,} | trainable after freeze: {trainable:,}")

## Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = None
best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}


def mixup_batch(images, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1.0 - lam) * images[idx]
    return mixed, labels, labels[idx], lam


def cutmix_batch(images, labels, alpha=CUTMIX_ALPHA):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(images.size(0), device=images.device)
    _, _, H, W = images.shape
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_ratio), int(H * cut_ratio)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = max(cx - cut_w // 2, 0), max(cy - cut_h // 2, 0)
    x2, y2 = min(cx + cut_w // 2, W), min(cy + cut_h // 2, H)
    mixed = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
    lam = 1.0 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, labels, labels[idx], lam


def mixup_loss(outputs, y_a, y_b, lam):
    return lam * criterion(outputs, y_a) + (1.0 - lam) * criterion(outputs, y_b)


def train_one_epoch(model, loader, use_mixup=True, scheduler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if use_mixup and (MIXUP_ALPHA > 0 or CUTMIX_ALPHA > 0):
            if CUTMIX_ALPHA > 0 and (MIXUP_ALPHA <= 0 or random.random() < 0.5):
                images, y_a, y_b, lam = cutmix_batch(images, labels)
            else:
                images, y_a, y_b, lam = mixup_batch(images, labels)
            outputs = model(images)
            loss = mixup_loss(outputs, y_a, y_b, lam)
            preds = outputs.argmax(dim=1)
            correct += (
                lam * (preds == y_a).float() + (1.0 - lam) * (preds == y_b).float()
            ).sum().item()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * images.size(0)
        total += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def save_checkpoint(model, epoch, val_acc):
    torch.save(
        {"model_state_dict": model.state_dict(), "epoch": epoch, "val_acc": val_acc},
        CKPT_PATH,
    )


def run_epochs(
    model,
    max_epochs,
    phase_name,
    lr=None,
    param_groups=None,
    use_mixup=True,
    build_scheduler=None,
    scheduler_step_per_epoch=False,
):
    global optimizer, best_val_acc, best_epoch, epochs_without_improvement

    if param_groups is not None:
        optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
        lr_msg = f"backbone lr={PHASE2_LR:.2e}, head lr={PHASE2_LR * PHASE2_HEAD_LR_MULT:.2e}"
    else:
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr,
            weight_decay=WEIGHT_DECAY,
        )
        lr_msg = f"lr={lr}"

    scheduler = build_scheduler(optimizer) if build_scheduler else None

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n--- {phase_name}: up to {max_epochs} epochs, {lr_msg}, trainable={n_trainable:,} ---")

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, use_mixup=use_mixup,
            scheduler=None if scheduler_step_per_epoch else scheduler
        )
        if scheduler is not None and scheduler_step_per_epoch:
            scheduler.step()
        val_loss, val_acc = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch}/{max_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            epochs_without_improvement = 0
            save_checkpoint(model, epoch, val_acc)
            print(f"  -> saved checkpoint (val acc {val_acc:.4f})")
        else:
            epochs_without_improvement += 1

        if EARLY_STOP_PATIENCE and epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"  -> no val improvement for {EARLY_STOP_PATIENCE} epochs; stopping {phase_name}")
            break


# Phase 1: frozen backbone, train head (no mixup)
epochs_without_improvement = 0
freeze_efficientnet_features(model)


def build_phase1_scheduler(opt):
    return torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=PHASE1_LR,
        total_steps=PHASE1_EPOCHS * len(train_loader),
        pct_start=0.2,
    )


run_epochs(
    model,
    PHASE1_EPOCHS,
    "Phase 1: head warm-up (features frozen)",
    lr=PHASE1_LR,
    use_mixup=False,
    build_scheduler=build_phase1_scheduler,
)

# Phase 2: full fine-tune, differential LR, cosine annealing + early stopping
epochs_without_improvement = 0
unfreeze_efficientnet_all(model)


def build_phase2_scheduler(opt):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=PHASE2_EPOCHS, eta_min=1e-7
    )


run_epochs(
    model,
    PHASE2_EPOCHS,
    "Phase 2: full fine-tune",
    param_groups=[
        {"params": model.features.parameters(), "lr": PHASE2_LR},
        {"params": model.classifier.parameters(), "lr": PHASE2_LR * PHASE2_HEAD_LR_MULT},
    ],
    use_mixup=True,
    build_scheduler=build_phase2_scheduler,
    scheduler_step_per_epoch=True,
)

if os.path.isfile(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(
        f"\nLoaded best checkpoint: epoch {ckpt['epoch']} | val acc {ckpt['val_acc']:.4f}"
    )

print(f"Done. Best val acc: {best_val_acc:.4f} at epoch {best_epoch}")

## Visualization

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="train")
axes[1].plot(epochs_range, history["val_acc"], label="val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training and Validation Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## Submission

Run the cell below to load `checkpoints/best_model_On_Kaggle.pt` and write `submission.csv` with 5-view test-time augmentation.


In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, test_dir, image_ids, transform=None):
        self.test_dir = test_dir
        self.image_ids = list(image_ids)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        fname = self.image_ids[idx]
        path = os.path.join(self.test_dir, fname)
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, fname


tta_tf_list = [
    val_tf,
    transforms.Compose([
        transforms.Resize((IMG_SIZE + CROP_PADDING // 2, IMG_SIZE + CROP_PADDING // 2)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + CROP_PADDING, IMG_SIZE + CROP_PADDING)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + CROP_PADDING * 2, IMG_SIZE + CROP_PADDING * 2)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + CROP_PADDING * 2, IMG_SIZE + CROP_PADDING * 2)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]


@torch.no_grad()
def predict_with_tta(model, image_ids):
    model.eval()
    prob_sums = {}

    for tf in tta_tf_list:
        loader = DataLoader(
            TestImageDataset(TEST_DIR, image_ids, transform=tf),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        for images, fnames in tqdm(loader, leave=False, desc="tta"):
            probs = torch.softmax(model(images.to(device)), dim=1).cpu().numpy()
            for fname, prob in zip(fnames, probs):
                if fname not in prob_sums:
                    prob_sums[fname] = np.zeros(NUM_CLASSES, dtype=np.float64)
                prob_sums[fname] += prob

    return {fname: int(np.argmax(prob)) for fname, prob in prob_sums.items()}


submission_df = pd.read_csv(SAMPLE_SUBMISSION)
sample_ids = set(submission_df["ID"].tolist())
extra_ids = sorted(
    [f for f in os.listdir(TEST_DIR) if f.endswith(".jpg") and f not in sample_ids],
    key=lambda x: int(x.split(".")[0]),
)
all_test_ids = submission_df["ID"].tolist() + extra_ids
print(f"Test images: {len(all_test_ids)}")

if not os.path.isfile(SUBMIT_CKPT_PATH):
    raise FileNotFoundError(f"Missing checkpoint: {SUBMIT_CKPT_PATH}")

checkpoint = torch.load(SUBMIT_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(
    f"Loaded {SUBMIT_CKPT_PATH} | epoch {checkpoint.get('epoch', '?')} "
    f"| val acc {checkpoint.get('val_acc', 0):.4f}"
)

predictions = predict_with_tta(model, all_test_ids)

out_df = pd.DataFrame({"ID": all_test_ids, "Label": [predictions[i] for i in all_test_ids]})
assert set(out_df["Label"]) <= set(range(NUM_CLASSES))

out_df.to_csv(SUBMISSION_OUT, index=False)
print(f"Wrote {SUBMISSION_OUT} ({len(out_df)} rows)")
print(out_df.head())
